In [ ]:
#@title 1. Install AI Engine
!pip install -q -U coqui-tts fastapi "uvicorn[standard]" python-multipart pyngrok nest-asyncio soundfile
print("Done!")

In [ ]:
#@title 2. Load XTTS-v2 Model
import os, json, torch
from TTS.api import TTS
os.environ["COQUI_TOS_AGREED"] = "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(DEVICE)
xtts_model = tts.synthesizer.tts_model
BASE_DIR = "/content/vox_ai_engine"
VOICES_DIR, OUTPUT_DIR = f"{BASE_DIR}/voices", f"{BASE_DIR}/outputs"
REGISTRY_PATH = f"{BASE_DIR}/voice_registry.json"
os.makedirs(VOICES_DIR, exist_ok=True); os.makedirs(OUTPUT_DIR, exist_ok=True)
if not os.path.exists(REGISTRY_PATH): json.dump({}, open(REGISTRY_PATH, "w"))
LATENT_CACHE = {}
print("Model Ready on", DEVICE)

In [ ]:
#@title 3. Setup Server
import uuid, shutil, soundfile as sf
from fastapi import FastAPI, UploadFile, File, Form, HTTPException, Request
from fastapi.responses import Response
app = FastAPI()
def load_reg(): return json.load(open(REGISTRY_PATH))
def save_reg(d): json.dump(d, open(REGISTRY_PATH, "w"))
def get_lat(sid, path):
    if sid not in LATENT_CACHE: LATENT_CACHE[sid] = xtts_model.get_conditioning_latents(audio_path=[path])
    return LATENT_CACHE[sid]
@app.get("/health")
async def health(): return {"status": "ok", "engine": "xtts-v2"}
@app.get("/api/voices")
async def voices(): return {"voices": [{"id": k, "name": v["name"]} for k, v in load_reg().items()]}
@app.post("/api/voices/register")
async def reg(name: str = Form(...), speaker_wav: UploadFile = File(...)):
    sid = f"vox_{uuid.uuid4().hex[:12]}"
    p = f"{VOICES_DIR}/{sid}.wav"
    with open(p, "wb") as b: shutil.copyfileobj(speaker_wav.file, b)
    get_lat(sid, p)
    r = load_reg(); r[sid] = {"name": name.strip(), "wav_path": p}; save_reg(r)
    return {"success": True, "speaker_id": sid}
@app.delete("/api/voices/{sid}")
async def dele(sid: str):
    r = load_reg(); r.pop(sid, None); LATENT_CACHE.pop(sid, None); save_reg(r)
    return {"success": True}
@app.post("/api/tts")
async def tts_api(request: Request):
    ct = request.headers.get("content-type", "")
    tw = None
    try:
        if "multipart/form-data" in ct:
            f = await request.form()
            text, lang, sid, up = str(f.get("text","")), str(f.get("language","en")), str(f.get("speaker_id","")), f.get("speaker_wav")
            if up and hasattr(up, "file"):
                tw = f"{OUTPUT_DIR}/ref_{uuid.uuid4().hex[:8]}.wav"
                with open(tw, "wb") as out_f: shutil.copyfileobj(up.file, out_f)
        else:
            d = await request.json()
            text, lang, sid = d.get("text",""), d.get("language","en"), d.get("speaker_id","")
        r = load_reg()
        rw = tw or r.get(sid, {}).get("wav_path")
        if not rw: raise HTTPException(404, "Voice not found")
        gl, se = get_lat(sid if (sid in r and not tw) else f"t_{uuid.uuid4().hex[:6]}", rw)
        out = xtts_model.inference(text=text, language=lang, gpt_cond_latent=gl, speaker_embedding=se)
        op = f"{OUTPUT_DIR}/o_{uuid.uuid4().hex[:8]}.wav"
        sf.write(op, out["wav"], 24000)
        data = open(op, "rb").read(); os.remove(op)
        return Response(content=data, media_type="audio/wav")
    finally:
        if tw and os.path.exists(tw): os.remove(tw)
print("API Ready!")

In [ ]:
#@title 4. Start Live Link
import threading, time, uvicorn, nest_asyncio
from pyngrok import ngrok, conf

NGROK_TOKEN = "3MU0lapwclkkaOKlnXL55EKEfDF_3WAyHaUhcSs2Mb5Za9AEw"

conf.get_default().auth_token = NGROK_TOKEN.strip()
ngrok.kill()
nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8002, log_level="warning"), daemon=True).start()
time.sleep(2)
print("COPY THIS URL:", ngrok.connect(8002, bind_tls=True).public_url)